In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Singularity Machine Learning - Classification：Multiverse Computing による Qiskit Function
> **Note:** * Qiskit Functions は、IBM Quantum&reg; Premium Plan・Flex Plan・On-Prem（IBM Quantum Platform API 経由）プランのユーザーのみが利用できる実験的機能です。現在プレビュー リリースの段階にあり、変更される場合があります。

## 概要
「Singularity Machine Learning - Classification」関数を使用すると、量子の専門知識を必要とせずに、実世界の機械学習問題を量子ハードウェア上で解くことができます。アンサンブル手法をベースにしたこの Application 関数は、ハイブリッド分類器です。初期のアンサンブル学習にはブースティング・バギング・スタッキングといった古典的な手法を活用し、その後、変分量子固有値ソルバー（VQE）や量子近似最適化アルゴリズム（QAOA）などの量子アルゴリズムを用いて、学習済みアンサンブルの多様性・汎化能力・全体的な複雑さを向上させます。

他の量子機械学習ソリューションとは異なり、この関数は QPU の量子ビット数に制限されることなく、数百万のサンプルと特徴量を持つ大規模データセットを処理できます。量子ビット数は学習できるアンサンブルのサイズのみを決定します。また、高い柔軟性を持ち、金融・医療・サイバーセキュリティなど幅広い分野の分類問題に適用可能です。
高次元・ノイズが多い・クラス不均衡といった古典的に困難な問題においても、一貫して高い精度を達成します。
![動作原理](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
この関数は、以下のような方々のために構築されています。
1. 量子機械学習を自社の製品やサービスに統合することで技術力強化を図る企業のエンジニアおよびデータサイエンティスト、
2. 量子機械学習の応用を探求し、分類タスクに量子コンピューティングを活用しようとしている量子研究所の研究者、
3. 機械学習などの講義で量子コンピューティングの優位性を示したい教育機関の学生および教員。

以下の例では、`create`・`list`・`fit`・`predict` などのさまざまな機能を紹介するとともに、非線形な決定境界のために特に困難とされる問題である、2 つの交差する半円からなる合成問題への使用方法を示します。


## 関数の説明
この Qiskit Function を使用すると、Singularity の量子強化アンサンブル分類器を使って 2 値分類問題を解くことができます。内部では、ラベル付きデータセット上でアンサンブル分類器を古典的に学習するハイブリッドアプローチを採用し、その後 IBM&reg; QPU 上の量子近似最適化アルゴリズム（QAOA）を用いて最大の多様性と汎化性能が得られるよう最適化します。ユーザーフレンドリーなインターフェースを通じて、要件に合わせて分類器を設定し、任意のデータセットで学習させ、未知のデータセットに対して予測を行えます。

汎用的な分類問題を解くには、以下の手順を実施してください。
1. データセットを前処理し、学習セットとテストセットに分割します。必要に応じて、学習セットをさらに学習セットと検証セットに分割できます。これには [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html) を使用できます。
2. 学習セットがクラス不均衡の場合は、[imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets) を使用してクラスを均衡化するためにリサンプリングできます。
3. カタログの `file_upload` メソッドに都度関連するパスを渡して、学習・検証・テストセットをそれぞれ関数のストレージにアップロードします。
4. 関数の `create` アクションを使用して量子分類器を初期化します。このアクションは、学習器の数と種類・正則化（ラムダ値）・レイヤー数・古典オプティマイザーの種類・量子バックエンドなどの最適化オプションといったハイパーパラメータを受け付けます。
5. 関数の `fit` アクションを使用して、ラベル付き学習セット（および該当する場合は検証セット）を渡し、量子分類器を学習セットで学習します。
6. 関数の `predict` アクションを使用して、未知のテストセットに対して予測を行います。
## アクションベースのアプローチ
この関数はアクションベースのアプローチを採用しています。アクションを使ってタスクを実行したり状態を変更したりできる仮想環境と捉えることができます。現在利用可能なアクションは、[list](#1-list)・[create](#2-create)・[delete](#3-delete)・[fit](#4-fit)・[predict](#5-predict)・[fit_predict](#6-fit-predict)・[create_fit_predict](#7-create-fit-predict) です。以下の例では `create_fit_predict` アクションを使用しています。

In [ ]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [1, 0, 0, 1, 0]
Probabilities (first five results):  [[0.16849563539001172, 0.8315043646099888], [0.8726393386620336, 0.12736066133796647], [0.795344837290717, 0.20465516270928288], [0.36822585748882725, 0.6317741425111725], [0.6656662698604361, 0.3343337301395641]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 7.945035696029663}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 82.41029238700867}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 77.3459484577179}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 71.27004957199097}}


### 1. List

The `list` action retrieves all stored classifiers in `*.pkl.tar` format from the shared data directory. You can also access the contents of this directory by using the `catalog.files()` method. In general, the list action searches for files with the `*.pkl.tar` extension in the shared data directory and returns them in a list format.

#### Inputs

|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | The name of the action from among `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` and `delete`. | Yes |

#### Usage

In [ ]:
job = singularity.run(action="list")

### 1. List

`list` アクションは、共有データディレクトリから `*.pkl.tar` 形式で保存されているすべての分類器を取得します。`catalog.files()` メソッドを使用してこのディレクトリの内容にアクセスすることもできます。一般に、list アクションは共有データディレクトリ内で `*.pkl.tar` 拡張子を持つファイルを検索し、リスト形式で返します。

#### 入力

|     名前    |    型     | 説明 |   必須  |
|-------------|-----------|------|---------|
| `action` | `str` | `create`・`list`・`fit`・`predict`・`fit_predict`・`create_fit_predict`・`delete` の中からアクション名を指定します。 | はい |

#### 使用方法

In [ ]:
job = singularity.run(
    action="create",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
)

### 2. Create
`create` アクションは、指定された `quantum_classifier` タイプの分類器を指定されたパラメータを使って作成し、共有データディレクトリに保存します。

> **Note:** 現在、この関数がサポートしているのは `QuantumEnhancedEnsembleClassifier` のみです。
#### 入力
|     名前    |    型     | 説明 | 必須  | デフォルト |
|-------------|-----------|------|-------|------------|
| `action` | `str` | `create`・`list`・`fit`・`predict`・`fit_predict`・`create_fit_predict`・`delete` の中からアクション名を指定します。 | はい | - |
| `name` | `str` | 量子分類器の名前（例：`spam_classifier`）。 | はい | - |
| `instance` | `str` | IBM インスタンス。 | はい | - |
| `backend_name` | `str` | IBM コンピュートリソース。デフォルトは `None` で、保留中のジョブ数が最も少ないバックエンドが使用されます。 | いいえ | `None` |
| `quantum_classifier` | `str` | 量子分類器のタイプ（`QuantumEnhancedEnsembleClassifier`）。 | いいえ | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | アンサンブル内の学習器の数。 | いいえ | `10` |
| `learners_types` | `list` | 学習器のタイプ。サポートされているタイプには `DecisionTreeClassifier`・`GaussianNB`・`KNeighborsClassifier`・`MLPClassifier`・`LogisticRegression` などがあります。各タイプの詳細は [scikit-learn のドキュメント](https://scikit-learn.org/stable/supervised_learning.html)を参照してください。 | いいえ | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | アンサンブル内の各学習器タイプの割合。 | いいえ | `[1.0]` |
| `learners_options` | `list` | アンサンブル内の各学習器タイプのオプション。選択した学習器タイプに対応するオプションの完全な一覧は [scikit-learn のドキュメント](https://scikit-learn.org/stable/supervised_learning.html)を参照してください。 | いいえ | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` または `list` | 使用する正則化のタイプ：`onsite` または `alpha`。`onsite` はオンサイト項を制御し、値が大きいほどスパースなアンサンブルになります。`alpha` は相互作用項とオンサイト項のトレードオフを制御し、値が小さいほどスパースなアンサンブルになります。リストを指定した場合、各タイプでモデルが学習され、最もパフォーマンスの高いものが選択されます。 | いいえ | `onsite` |
| `regularization` | `str` または `float` または `list` | 正則化の値。`regularization_type` が `onsite` の場合は `0` から `+inf` の範囲、`alpha` の場合は `0` から `1` の範囲で指定します。`auto` に設定すると、自動正則化が使用され、選択された分類器と総分類器の比率（`regularization_desired_ratio`）と正則化パラメータの上限（`regularization_upper_bound`）を指定した二分探索によって最適な正則化パラメータが求められます。リストを指定した場合、各値でモデルが学習され、最もパフォーマンスの高いものが選択されます。 | いいえ | `0.01` |
| `regularization_desired_ratio` | `float` または `list` | 自動正則化における、選択された分類器と総分類器の目標比率。リストを指定した場合、各比率でモデルが学習され、最もパフォーマンスの高いものが選択されます。 | いいえ | `0.75` |
| `regularization_upper_bound` | `float` または `list` | 自動正則化使用時の正則化パラメータの上限。リストを指定した場合、各上限値でモデルが学習され、最もパフォーマンスの高いものが選択されます。 | いいえ | `200` |
| `weight_update_method` | `str` | サンプルウェイトの更新方法：`logarithmic`（対数）または `quadratic`（二乗）。 | いいえ | `logarithmic` |
| `sample_scaling` | `boolean` | サンプルスケーリングを適用するかどうか。 | いいえ | `False` |
| `prediction_scaling` | `float` | 予測のスケーリング係数。 | いいえ | `None` |
| `optimizer_options` | `dictionary` | QAOA オプティマイザーのオプション。利用可能なオプションの一覧は本ドキュメントで後述します。 | いいえ | ... |
| `voting` | `str` | 学習器の予測・確率を集約する際に多数決（`hard`）または確率の平均（`soft`）を使用します。 | いいえ | `hard` |
| `prob_threshold` | `float` | 最適確率のしきい値。 | いいえ | `0.5` |
| `random_state` | `integer` | 再現性のためにランダム性を制御します。 | いいえ | `None` |


- また、`optimizer_options` は以下の通りです。

|     名前    |    型     | 説明 | 必須  | デフォルト |
|-------------|-----------|------|-------|------------|
| `num_solutions` | `integer` | 解の数 | いいえ | `1024` |
| `reps` | `integer` | 繰り返し回数 | いいえ | `4` |
| `sparsify` | `float` | スパース化のしきい値 | いいえ | `0.001` |
| `theta` | `float` | QAOA の変分パラメータ theta の初期値 | いいえ | `None` |
| `simulator` | `boolean` | シミュレーターを使用するか QPU を使用するか | いいえ | `False` |
| `classical_optimizer` | `str` | QAOA の古典オプティマイザー名。SciPy が提供するすべてのソルバー（[こちら](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize)に一覧があります）が使用可能です。それに合わせて `classical_optimizer_options` を設定する必要があります。 | いいえ | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | 古典オプティマイザーのオプション。利用可能なオプションの完全な一覧は [SciPy のドキュメント](https://docs.scipy.org/doc/scipy/reference/)を参照してください。 | いいえ | `{"maxiter": 60}` |
| `optimization_level` | `integer` | QAOA 回路の深さ | いいえ | `3` |
| `num_transpiler_runs` | `integer` | トランスパイラーの実行回数 | いいえ | `30` |
| `pass_manager_options` | `dictionary` | プリセットパスマネージャー生成のオプション | いいえ | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | Estimator のオプション。利用可能なオプションの完全な一覧は [Qiskit Runtime Client のドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options)を参照してください。 | いいえ | `None` |
| `sampler_options` | `dictionary` | Sampler のオプション。利用可能なオプションの完全な一覧は [Qiskit Runtime Client のドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)を参照してください。 | いいえ | `None` |

- `estimator_options` のデフォルト値は以下の通りです。

|     名前    |    型     | 値  |
|-------------|-----------|-----|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- `sampler_options` のデフォルト値は以下の通りです。

|     名前    |    型     | 値 |
|-------------|-----------|-----|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### 使用方法

In [ ]:
job = singularity.run(
    action="delete",
    name="classifier_name",  # specify the name of the classifier to delete here
)

#### バリデーション
- `name`:
    - 名前は一意である必要があり、最大 64 文字の文字列です。
    - 英数字とアンダースコアのみ使用できます。
    - 文字で始まり、アンダースコアで終わってはなりません。
    - 共有データディレクトリに同じ名前の分類器が既に存在していてはなりません。
### 3. Delete
`delete` アクションは、共有データディレクトリから分類器を削除します。

#### 入力
|     名前    |    型     | 説明 |   必須  |
|-------------|-----------|------|---------|
| `action`    | `str`    | アクション名。`delete` である必要があります。 | はい |
| `name`      | `str`    | 削除する分類器の名前。 | はい |

#### 使用方法

In [ ]:
job = singularity.run(
    action="fit",
    name="classifier_name",  # specify the name of the classifier to train here
    X=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
)

#### バリデーション
- `name`:
    - 名前は一意である必要があり、最大 64 文字の文字列です。
    - 英数字とアンダースコアのみ使用できます。
    - 文字で始まり、アンダースコアで終わってはなりません。
    - 共有データディレクトリに同じ名前の分類器が既に存在している必要があります。
### 4. Fit
`fit` アクションは、提供された学習データを使って分類器を学習します。

#### 入力
|     名前    |    型     | 説明 |   必須  |
|-------------|-----------|------|---------|
| `action`    | `str`    | アクション名。`fit` である必要があります。 | はい |
| `name`      | `str`    | 学習する分類器の名前。 | はい |
| `X`         | `array` または `list` または `str` | 学習データ。NumPy 配列・リスト、または共有データディレクトリ内のファイル名を参照する文字列を指定できます。 | はい |
| `y`         | `array` または `list` または `str` | 学習の目標値。NumPy 配列・リスト、または共有データディレクトリ内のファイル名を参照する文字列を指定できます。 | はい |
| `fit_params`| `dictionary`| 分類器の `fit` メソッドに渡す追加パラメータ。 | いいえ |

##### fit_params
|     名前    |    型     | 説明 |   必須  |   デフォルト   |
|-------------|-----------|------|---------|---------------|
| `validation_data` | `tuple` | 検証データとラベル。 | いいえ | `None` |
| `pos_label` | `integer` または `str` | 1 にマッピングするクラスラベル。 | いいえ | `None` |
| `optimization_data` | `str` | アンサンブルを最適化するデータセット。`train`・`validation`・`both` のいずれかを指定できます。 | いいえ | `train` |

#### 使用方法

In [ ]:
job = singularity.run(
    action="predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X=X_test,  # or "X_test.npy" if you uploaded it to the shared data directory
    options={
        "out": "output.json",
    },
)

#### バリデーション
- `name`:
    - 名前は一意である必要があり、最大 64 文字の文字列です。
    - 英数字とアンダースコアのみ使用できます。
    - 文字で始まり、アンダースコアで終わってはなりません。
    - 共有データディレクトリに同じ名前の分類器が既に存在している必要があります。
### 5. Predict
`predict` アクションは、ハードおよびソフト予測（確率）を取得するために使用します。

#### 入力
|     名前    |    型     | 説明 |   必須  |
|-------------|-----------|------|---------|
| `action`    | `str`    | アクション名。`predict` である必要があります。 | はい |
| `name`      | `str`    | 使用する分類器の名前。 | はい |
| `X`         | `array` または `list` または `str` | テストデータ。NumPy 配列・リスト、または共有データディレクトリ内のファイル名を参照する文字列を指定できます。 | はい |
| `options["out"]` | `str` | 予測を共有データディレクトリに保存する出力 JSON ファイル名。指定しない場合、予測はジョブの結果として返されます。 | いいえ |

#### 使用方法

In [ ]:
job = singularity.run(
    action="fit_predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "out": "output.json",
    },
)

#### バリデーション
- `name`:
    - 名前は一意である必要があり、最大 64 文字の文字列です。
    - 英数字とアンダースコアのみ使用できます。
    - 文字で始まり、アンダースコアで終わってはなりません。
    - 共有データディレクトリに同じ名前の分類器が既に存在している必要があります。
- `options["out"]`:
    - ファイル名は一意である必要があり、最大 64 文字の文字列です。
    - 英数字とアンダースコアのみ使用できます。
    - 文字で始まり、アンダースコアで終わってはなりません。
    - `.json` 拡張子が必要です。
### 6. Fit-predict
`fit_predict` アクションは、学習データを使って分類器を学習し、その後ハードおよびソフト予測（確率）を取得します。

#### 入力
|     名前    |    型     | 説明 |   必須  |
|-------------|-----------|------|---------|
| `action`    | `str`    | アクション名。`fit_predict` である必要があります。 | はい |
| `name`      | `str`    | 使用する分類器の名前。 | はい |
| `X_train`   | `array` または `list` または `str` | 学習データ。NumPy 配列・リスト、または共有データディレクトリ内のファイル名を参照する文字列を指定できます。 | はい |
| `y_train`   | `array` または `list` または `str` | 学習の目標値。NumPy 配列・リスト、または共有データディレクトリ内のファイル名を参照する文字列を指定できます。 | はい |
| `X_test`    | `array` または `list` または `str` | テストデータ。NumPy 配列・リスト、または共有データディレクトリ内のファイル名を参照する文字列を指定できます。 | はい |
| `fit_params`| `dictionary`| 分類器の `fit` メソッドに渡す追加パラメータ。 | いいえ |
| `options["out"]` | `str` | 予測を共有データディレクトリに保存する出力 JSON ファイル名。指定しない場合、予測はジョブの結果として返されます。 | いいえ |

#### 使用方法

In [ ]:
job = singularity.run(
    action="create_fit_predict",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "save": True,
        "out": "output.json",
    },
)

#### バリデーション
- `name`:
    - 名前は一意である必要があり、最大 64 文字の文字列です。
    - 英数字とアンダースコアのみ使用できます。
    - 文字で始まり、アンダースコアで終わってはなりません。
    - 共有データディレクトリに同じ名前の分類器が既に存在している必要があります。

- `options["out"]`:
    - ファイル名は一意である必要があり、最大 64 文字の文字列です。
    - 英数字とアンダースコアのみ使用できます。
    - 文字で始まり、アンダースコアで終わってはなりません。
    - `.json` 拡張子が必要です。
### 7. Create-fit-predict
`create_fit_predict` アクションは、分類器を作成し、提供された学習データを使って学習した後、ハードおよびソフト予測（確率）を取得します。

#### 入力
| 名前 | 型 | 説明 | 必須 |
|------|----|------|------|
| `action` | `str` | `create`・`list`・`fit`・`predict`・`fit_predict`・`create_fit_predict`・`delete` の中からアクション名を指定します。 | はい |
| `name` | `str` | 使用する分類器の名前。 | はい |
| `quantum_classifier` | `str` | 分類器のタイプ（`QuantumEnhancedEnsembleClassifier`）。デフォルトは `QuantumEnhancedEnsembleClassifier` です。 | いいえ |
| `X_train` | `array` または `list` または `str` | 学習データ。NumPy 配列・リスト、または共有データディレクトリ内のファイル名を参照する文字列を指定できます。 | はい |
| `y_train` | `array` または `list` または `str` | 学習の目標値。NumPy 配列・リスト、または共有データディレクトリ内のファイル名を参照する文字列を指定できます。 | はい |
| `X_test` | `array` または `list` または `str` | テストデータ。NumPy 配列・リスト、または共有データディレクトリ内のファイル名を参照する文字列を指定できます。 | はい |
| `fit_params` | `dictionary` | 分類器の `fit` メソッドに渡す追加パラメータ。 | いいえ |
| `options["save"]` | `boolean` | 学習済み分類器を共有データディレクトリに保存するかどうか。デフォルトは `True` です。 | いいえ |
| `options["out"]` | `str` | 予測を共有データディレクトリに保存する出力 JSON ファイル名。指定しない場合、予測はジョブの結果として返されます。 | いいえ |

#### 使用方法

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

#### バリデーション
- `name`:
    - `options["save"]` が `True` に設定されている場合：
        - 名前は一意である必要があり、最大 64 文字の文字列です。
        - 英数字とアンダースコアのみ使用できます。
        - 文字で始まり、アンダースコアで終わってはなりません。
        - 共有データディレクトリに同じ名前の分類器が既に存在していてはなりません。

- `options["out"]`:
    - ファイル名は一意である必要があり、最大 64 文字の文字列です。
    - 英数字とアンダースコアのみ使用できます。
    - 文字で始まり、アンダースコアで終わってはなりません。
    - `.json` 拡張子が必要です。
---

## はじめに
[IBM Quantum Platform APIキー](http://quantum.cloud.ibm.com/)を使用して認証し、以下のようにQiskit Functionを選択してください。

In [3]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[-0.99958218  0.02890441]
 [ 0.03285169  0.24578719]
 [ 1.13127903 -0.49134546]
 [ 1.86951286  0.00608971]
 [ 0.20190413  0.97940529]
 [ 0.8831311   0.46912627]
 [-0.10819442  0.99412975]
 [-0.20005727  0.97978421]
 [-0.78775705  0.61598607]
 [ 1.82453236 -0.0658148 ]]
Targets: [0 1 1 1 0 0 0 0 0 1]


## 使用例
この例では、「Singularity Machine Learning - Classification」ファンクションを使用して、2つの互い違いに絡み合った三日月形の半円から構成されるデータセットを分類します。このデータセットは合成的かつ2次元であり、バイナリラベルが付与されています。重心ベースのクラスタリングや線形分類などのアルゴリズムに対して難しいデータセットとして作成されています。
![Moonsデータセット](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
このプロセスを通じて、分類器の作成、トレーニングデータへのフィッティング、テストデータへの予測、および終了後の分類器の削除方法を学びます。
開始前に、[scikit-learn](https://scikit-learn.org/)をインストールする必要があります。以下のコマンドを使用してインストールしてください。

In [4]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


以下の手順を実行してください。

1) [scikit-learn](https://scikit-learn.org/)の[make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html)関数を使用して合成データセットを作成します。
2) 生成した合成データセットを[共有データディレクトリ](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html)にアップロードします。
3) [create](#2-create)アクションを使用して量子強化分類器を作成します。
4) [list](#1-list)アクションを使用して分類器の一覧を表示します。
5) [fit](#4-fit)アクションを使用して、トレーニングデータで分類器を学習させます。
6) [predict](#5-predict)アクションを使用して、学習済み分類器でテストデータを予測します。
7) [delete](#3-delete)アクションを使用して分類器を削除します。
8) 完了後にクリーンアップを行います。

**ステップ1.** 必要なモジュールをインポートして合成データセットを生成し、トレーニングデータセットとテストデータセットに分割します。

In [5]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [6]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}
['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar', 'my_classifier.pkl.tar']


**ステップ2.** ラベル付きのトレーニングデータセットとテストデータセットをローカルディスクに保存し、[共有データディレクトリ](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html)にアップロードします。

In [7]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 8.45469617843628}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 69.4949426651001}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 73.01881957054138}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 75.4787163734436}}}}


**Step 5.** Obtain predictions and probabilities from the quantum-enhanced classifier using the [predict](#5-predict) action.

In [8]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 1, 0, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0], [0.0, 1.0]]


**ステップ3.** [create](#2-create)アクションを使用して量子強化分類器を作成します。

In [9]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [ ]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

## Benchmarks

These benchmarks show that the classifier can achieve extremely high accuracies on challenging problems. They also show that increasing the number of learners in the ensemble (number of qubits) can lead to increased accuracy.

"Classical accuracy" refers to the accuracy obtained using corresponding classical state of the art which, in this case, is an AdaBoost classifier based on an ensemble of size 75. "Quantum accuracy", on the other hand, refers to the accuracy obtained using the "Singularity Machine Learning - Classification".

| Problem | Dataset Size | Ensemble Size | Number of Qubits | Classical Accuracy | Quantum Accuracy | Improvement |
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Grid stability | 5000 examples, 12 features | 55 | 55 |  76% | 91% | 15% |
| Grid stability | 5000 examples, 12 features | 65 | 65 |  76% | 92% | 16% |
| Grid stability | 5000 examples, 12 features | 75 | 75 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 85 | 85 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 100 | 100 |  76% | 95% | 19% |

----

As quantum hardware evolves and scales, the implications for our quantum classifier become increasingly significant. While the number of qubits does impose limitations on the size of the ensemble that can be utilized, it does not restrict the volume of data that can be processed. This powerful capability enables the classifier to efficiently handle datasets containing millions of data points and thousands of features. Importantly, the constraints related to ensemble size can be addressed through the implementation of a large-scale version of the classifier. By leveraging an iterative outer-loop approach, the ensemble can be dynamically expanded, enhancing flexibility and overall performance. However, it's worth noting that this feature has not yet been implemented in the current version of the classifier.

## Changelog

### 4 June 2025
- Upgraded `QuantumEnhancedEnsembleClassifier` with the following updates:
  - Added onsite/alpha regularization. You can specify `regularization_type` to be `onsite` or `alpha`
  - Added auto-regularization. You can set `regularization` to `auto` to use auto-regularization
  - Added `optimization_data` parameter to the `fit` method to choose optimization data for quantum optimization. You can use one of these options: `train`, `validation`, or `both`
  - Improved overall performance
- Added detailed status tracking for running jobs

### 20 May 2025
- Standardized error handling

### 18 March 2025
- Upgraded qiskit-serverless to 0.20.0 and base image to 0.20.1

### 14 February 2025
- Upgraded base image to 0.19.1

### 6 February 2025
- Upgraded qiskit-serverless to 0.19.0 and base image to 0.19.0

### 13 November 2024
- Release of Singularity Machine Learning - Classification

## Get support

For any questions, [reach out to Multiverse Computing](mailto:singularity@multiversecomputing.com).

Be sure to include the following information:

- The Qiskit Function Job ID (`job.job_id`)
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Multiverse Computing's Singularity Machine Learning Classification function](https://quantum.cloud.ibm.com/functions?id=multiverse-singularity).
- Review [Leclerc, L., et al. (2023). Financial risk management on a neutral atom quantum processor. Physical Review Research, 5, 043117.](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.5.043117)

</Admonition>